In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 27. Flash Attention

> **学習目標**
> - Flash Attention (GPU SRAM vs HBM)
> - IO 複雑度 (tiling)
> -

## 27.1 GPU

GPU :
- **SRAM** (, , ~20MB)
- **HBM** (High Bandwidth Memory, "GPU " , ~40-80GB)
- **DRAM** (CPU , , )

HBM ↔ SRAM . Attention $n \times n$ 行列 HBM → .

## 27.2 Attention

$$S = QK^\top \in \mathbb{R}^{n \times n}$$ (HBM )
$$P = \mathrm{softmax}(S) \in \mathbb{R}^{n \times n}$$ (HBM )
$$O = PV \in \mathbb{R}^{n \times d}$$ (HBM )

- HBM : $O(n^2)$ 
- SRAM :

## 27.3 Flash Attention — Online Softmax

** **: $n \times n$ 行列 , SRAM .

### Online Softmax
 計算 , $\max$ $\sum$ 計算:
$$m_i^{(j)} = \max(m_i^{(j-1)}, \max_j S_{ij})$$
$$\ell_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} \ell_i^{(j-1)} + \sum_j e^{S_{ij} - m_i^{(j)}}$$

### Tiling
$Q, K, V$ SRAM , Attention 計算 結果 HBM .

**IO 複雑度**:
- : $O(n^2)$ HBM
- Flash: $O(n^2 d / M)$ where $M$ = SRAM →

### バックプロパゲーション 計算
バックプロパゲーション $S, P$ 計算 ( $O(n)$ vs $O(n^2)$).


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

#  Attention
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-1, -2) / (d_k ** 0.5)  # HBM 
    P = F.softmax(S, dim=-1)  # HBM /
    O = P @ V  # HBM /
    return O

# Flash Attention  (, online softmax)
def flash_attention_naive(Q, K, V, block_size=64):
    """Flash attention    ."""
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)
    # Q Block, K/V  ( Flash   Block)
    for i in range(0, N, block_size):
        Q_block = Q[:, :, i:i+block_size, :]  # (B, H, bs, D)
        S_block = Q_block @ K.transpose(-1, -2) / (D ** 0.5)  # (B, H, bs, N)
        P_block = F.softmax(S_block, dim=-1)
        O[:, :, i:i+block_size, :] = P_block @ V
    return O

#  
torch.manual_seed(0)
B, H, N, D = 1, 2, 128, 32
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O_std = standard_attention(Q, K, V)
O_flash = flash_attention_naive(Q, K, V, block_size=32)
print(f" vs Flash  誤差: {(O_std - O_flash).abs().max():.2e}")
print("=> Result . Flash Memory Efficiency 線.")


## 27.4 PyTorch SDPA

PyTorch `scaled_dot_product_attention` :
- `flash`: Flash Attention v2 (GPU)
- `mem_efficient`: Memory-Efficient Attention (GPU/CPU)
- `math`:


In [ ]:
# SDPA  比較
import time

#     
def bench_attention(n, d=64, n_heads=8, device='cpu'):
    """Attention Time Memory Measurement."""
    Q = torch.randn(1, n_heads, n, d, device=device)
    K = torch.randn(1, n_heads, n, d, device=device)
    V = torch.randn(1, n_heads, n, d, device=device)

    # Standard Implementation
    def std_attn():
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

    # SDPA
    def sdpa_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    # Time Measurement
    for _ in range(2): std_attn()  # warmup
    t0 = time.perf_counter()
    for _ in range(3):
        std_attn()
    t_std = (time.perf_counter() - t0) / 3 * 1000

    for _ in range(2): sdpa_attn()
    t0 = time.perf_counter()
    for _ in range(3):
        sdpa_attn()
    t_sdpa = (time.perf_counter() - t0) / 3 * 1000

    return t_std, t_sdpa

print(f"{'n':>6} | {'Standard (ms)':>15} | {'SDPA (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n in [256, 512, 1024, 2048]:
    t_std, t_sdpa = bench_attention(n, device='cpu')
    print(f"{n:>6} | {t_std:>15.3f} | {t_sdpa:>12.3f} | {t_std/t_sdpa:>9.2f}x")


## 27.5

 Attention $n \times n$ Attention 行列 → $O(n^2)$ .

Flash Attention Attention 行列 → $O(n)$ .

: $n = 8192, h = 32, d_k = 128, B = 1$:
- : $B \cdot h \cdot n^2 \cdot 4 = 8$ GB (FP32)
- Flash: $O(n)$ → 100MB


In [ ]:
#  
def attention_memory_gb(n, n_heads, d_k, batch=1, dtype_bytes=4):
    """Standard Attention Attention Matrix Memory."""
    return batch * n_heads * n * n * dtype_bytes / (1024**3)

print(f"Attention Matrix Memory (FP32):")
print(f"{'n':>6} | {'Memory (GB)':>12} | {'Flash (Estimation)':>14}")
print("-" * 40)
for n in [1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(n, n_heads=32, d_k=128)
    flash_mem = n * 32 * 128 * 4 / (1024**3)  #  O(n)
    print(f"{n:>6} | {mem:>12.3f} | {flash_mem:>14.4f}")
print("\n=> n=8192 Standard 8GB, Flash  MB. Difference 複雑度.")


## 27.6 Flash Attention v2/v3

- **v1** (Dao et al., 2022): , online softmax
- **v2** (2023): , バックプロパゲーション
- **v3** (2024): Hopper (H100) ,

## 27.7 Ring Attention — GPU

$Q, K, V$ GPU , GPU Attention.
- GPU
- 1M+

## 27.8 要点

| | IO 複雑度 | | 速度 |
|---|---|---|---|
| Attention | $O(n^2)$ | $O(n^2)$ | |
| Flash Attention v2 | $O(n^2 d/M)$ | $O(n)$ | |
| Flash Attention v3 | | $O(n)$ | H100 |
| Ring Attention | | $O(n/k)$ (k GPU) | |

## 演習問題

1. PyTorch SDPA `math`, `mem_efficient` 比較.
2. シーケンス長 1024, 4096, 16384 vs SDPA 時間 比較.
3. Online Softmax 結果 .
4. Flash Attention バックプロパゲーション .
5. Ring Attention GPU .

> 解答: `solutions/ch27_solutions.ipynb`
